# [7.1] Logit Lens, Tuned Lens, and Patchscopes - Solutions

This notebook runs the solved activation-to-language contracts, then displays the report-backed `gelu-1l` signature result. Keep the claim boundary in view: this is a local mechanics preflight, not a released tuned-lens benchmark or proof that decoded text is the model's belief.

<details>
<summary>Expected output</summary>

The local tests should all print pass messages. The final table and plots should match the committed `verification_report.json`: tuned lens `0.450` vs logit lens `0.075`, Patchscope `1.000` vs text-only `0.000`, random max confidence about `0.0407`, and peak VRAM under `1 GB`.

</details>

<details>
<summary>Help - why these controls matter</summary>

Activation-to-language tools can be persuasive even when they are only reading prompt priors or decoder artifacts. Held-out validation, text-only baselines, counterfactuals, and random activations each remove a different shortcut.

</details>


In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter7_activation_to_language"
section = "part1_lenses_patchscopes"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_lenses_patchscopes.tests as tests
import part1_lenses_patchscopes.solutions as solutions

LensAccuracyReport = solutions.LensAccuracyReport
PatchscopeAccuracyReport = solutions.PatchscopeAccuracyReport
CounterfactualActivationReport = solutions.CounterfactualActivationReport
RandomActivationConfidenceReport = solutions.RandomActivationConfidenceReport
logit_lens = solutions.logit_lens
top_tokens = solutions.top_tokens
tuned_lens = solutions.tuned_lens
prediction_accuracy = solutions.prediction_accuracy
lens_accuracy_report = solutions.lens_accuracy_report
attention_lens = solutions.attention_lens
patchscope_prompt = solutions.patchscope_prompt
patchscope_accuracy_report = solutions.patchscope_accuracy_report
replace_final_position_activation = solutions.replace_final_position_activation
counterfactual_activation_report = solutions.counterfactual_activation_report
random_activation_confidence_report = solutions.random_activation_confidence_report
run_smoke_test = solutions.run_smoke_test


## Local Lens Tests

These tests cover logit lens projection, top-token reporting, tuned-lens correction, prediction accuracy, and attention-weighted value decoding.


In [ ]:
tests.test_logit_lens_and_top_tokens_match_reference(logit_lens, top_tokens)
tests.test_top_tokens_rejects_invalid_k(top_tokens)
tests.test_tuned_lens_improves_over_logit_lens_on_toy_targets(
    logit_lens,
    tuned_lens,
    lens_accuracy_report,
)
tests.test_tuned_lens_uses_bias_and_leading_dims(tuned_lens, prediction_accuracy)
tests.test_prediction_accuracy_rejects_shape_mismatch(prediction_accuracy)
tests.test_attention_lens_decodes_attention_weighted_values(attention_lens)
tests.test_attention_lens_rejects_rank_or_key_mismatch(attention_lens)


## Patchscope And Control Tests

These tests cover template strings, text-only baselines, final-position activation replacement, counterfactual answer changes, random-activation confidence, and the whole smoke-test contract.


In [ ]:
tests.test_patchscope_templates_and_accuracy_report(
    patchscope_prompt,
    patchscope_accuracy_report,
    replace_final_position_activation,
)
tests.test_replace_final_position_activation_rejects_bad_shapes(
    replace_final_position_activation,
)
tests.test_counterfactual_and_random_activation_controls(
    counterfactual_activation_report,
    random_activation_confidence_report,
)
tests.test_notebook_contract(run_smoke_test)
contract = run_smoke_test(cpu=True)
contract


## Signature Result

<details>
<summary>Interpreting the signature result</summary>

The report validates a pinned TransformerLens `gelu-1l` path. The tuned lens improves held-out agreement with the model's own final predictions; Patchscope activation insertion beats the neutral text-only prompt; counterfactual and random-activation controls pass. This is scoped evidence for the mechanics, not a general semantic decoder claim.

</details>


In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def signature_table(gpu: dict) -> list[tuple[str, object]]:
    return [
        ("model", f"{gpu['model_name']} / {gpu['hf_model_id']}"),
        ("HF revision", gpu["hf_revision"][:12]),
        ("train / held-out prompts", f"{gpu['train_prompt_count']} / {gpu['heldout_prompt_count']}"),
        ("held-out positions", gpu["heldout_position_count"]),
        ("logit-lens accuracy", round(gpu["logit_lens_accuracy"], 3)),
        ("tuned-lens accuracy", round(gpu["tuned_lens_accuracy"], 3)),
        ("tuned-lens improvement", round(gpu["tuned_lens_improvement"], 3)),
        ("final decode max abs error", f"{gpu['final_decode_max_abs_error']:.2e}"),
        ("Patchscope hook", gpu["patchscope_hook_name"]),
        ("Patchscope / text-only accuracy", f"{gpu['patchscope_accuracy']:.3f} / {gpu['text_only_accuracy']:.3f}"),
        ("patched min target margin", round(gpu["patchscope_min_patched_target_margin"], 4)),
        ("text-only max target margin", round(gpu["patchscope_max_text_only_target_margin"], 3)),
        ("counterfactual decoded token", f"{gpu['counterfactual_original_token']!r} -> {gpu['counterfactual_patched_token']!r}"),
        ("random max confidence", round(gpu["random_max_confidence"], 4)),
        ("attention lens shape", gpu["attention_lens_logits_shape"]),
        ("peak VRAM GB", round(gpu["peak_vram_gb"], 3)),
    ]


gpu = run_gpu_test(max_vram_gb=24.0)
signature_table(gpu)


In [ ]:
import matplotlib.pyplot as plt

gpu = run_gpu_test(max_vram_gb=24.0)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))

axes[0].bar(
    ["logit", "tuned"],
    [gpu["logit_lens_accuracy"], gpu["tuned_lens_accuracy"]],
    color=["#2563eb", "#16a34a"],
)
axes[0].set_title("Held-out lens decoding")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("top-1 agreement")

axes[1].bar(
    ["text-only", "patched"],
    [gpu["text_only_accuracy"], gpu["patchscope_accuracy"]],
    color=["#94a3b8", "#7c3aed"],
)
axes[1].set_title("Patchscope target recovery")
axes[1].set_ylim(0, 1)

axes[2].bar(
    ["decode err", "random conf", "patch margin"],
    [
        gpu["final_decode_max_abs_error"] / 1e-4,
        gpu["random_max_confidence"] / 0.1,
        gpu["patchscope_min_patched_target_margin"] / 0.01,
    ],
    color=["#0f766e", "#f97316", "#16a34a"],
)
axes[2].axhline(1.0, color="#334155", linewidth=1, linestyle="--")
axes[2].set_title("Control thresholds")
axes[2].set_ylabel("actual / threshold")
axes[2].tick_params(axis="x", rotation=15)

fig.tight_layout()
plt.show()


## Limitations

The report proves a scoped local preflight on one small public checkpoint and small safe prompt sets. It does not establish human semantic ground truth, broad tuned-lens quality, layerwise Patchscope robustness, generated-completion behavior, or that decoded text is literally the model's belief.
